In [4]:
import cv2
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
# MTCNN sẽ tìm vị trí khuôn mặt trong ảnh
detector = MTCNN()

# FaceNet dùng để chuyển khuôn mặt -> vector đặc trưng (embedding 512 chiều)
embedder = FaceNet()

In [ ]:
# FUNCTION: detect & crop face
def extract_face(img, required_size=(160,160)):
    """
    Phát hiện khuôn mặt trong ảnh và cắt ra đúng kích thước chuẩn cho FaceNet
    img : numpy array -> ảnh đọc bằng cv2
    required_size : tuple -> kích thước ảnh đầu ra (FaceNet yêu cầu 160x160)
    
    Returns
    face_array : numpy array hoặc None
        trả về ảnh khuôn mặt đã resize
        nếu không detect được mặt → trả về None
    """

    # MTCNN dùng ảnh RGB, trong khi OpenCV đọc ảnh dạng BGR
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # detect khuôn mặt
    results = detector.detect_faces(img_rgb)

    # nếu không phát hiện khuôn mặt
    if len(results) == 0:
        return None

    # lấy bounding box khuôn mặt đầu tiên
    x, y, w, h = results[0]['box']
    
    # tránh lỗi toạ độ âm
    x, y = abs(x), abs(y)
    
    # crop khuôn mặt từ ảnh gốc
    face = img_rgb[y:y+h, x:x+w]
    
    # resize về kích thước chuẩn cho FaceNet
    face = cv2.resize(face, required_size)
    
    return face

In [11]:
# FUNCTION: face -> embedding
def get_embedding(face_img):
    """
    Chuyển ảnh khuôn mặt -> vector đặc trưng (embedding)
    embedding là vector 512 chiều đại diện cho khuôn mặt dùng để so sánh độ giống giữa 2 người
    face_img : numpy array
        ảnh khuôn mặt đã crop (160x160)
    
    Returns
    embedding : numpy array (512,) vector đặc trưng của khuôn mặt
    """

    # chuyển kiểu dữ liệu sang float32
    face_img = face_img.astype('float32')

    # chuẩn hoá pixel về khoảng [0,1]
    face_img = face_img / 255.0

    # thêm chiều batch vì model yêu cầu input dạng (batch_size, h, w, c)
    samples = np.expand_dims(face_img, axis=0)

    # tạo vector embedding
    embedding = embedder.embeddings(samples)

    # trả về vector 1 chiều (512)
    return embedding[0]


In [16]:
# LOAD REFERENCE IMAGE
img_path = "face_image_set/test/reference.jpg"
# đọc ảnh bằng OpenCV
img = cv2.imread(img_path)

# kiểm tra ảnh có load được không
if img is None:
    print("Không tìm thấy ảnh, kiểm tra lại đường dẫn")
else:
    # detect và crop khuôn mặt
    face = extract_face(img)

    # kiểm tra có detect được mặt không
    if face is None:
        print("Không phát hiện khuôn mặt trong ảnh mẫu")
    else:
        # tạo vector embedding cho ảnh mẫu
        reference_embedding = get_embedding(face)

        print("Reference embedding created")
        print("Embedding shape:", reference_embedding.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Reference embedding created
Embedding shape: (512,)


In [19]:
# =============================
# REALTIME FACE RECOGNITION
# + SHOW MATCHED IMAGE
# =============================

# mở webcam
cap = cv2.VideoCapture(0)

threshold = 0.6

# chuẩn bị ảnh reference để hiển thị
reference_display = cv2.imread(img_path)

print("Press Q to quit")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    face = extract_face(frame)

    if face is not None:

        live_embedding = get_embedding(face)

        similarity = cosine_similarity(
            [reference_embedding],
            [live_embedding]
        )[0][0]

        # detect bounding box
        results = detector.detect_faces(
            cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        )

        if len(results) > 0:

            x, y, w, h = results[0]['box']
            x, y = abs(x), abs(y)

            # nếu match
            if similarity > threshold:

                label = "Matched"
                color = (0,255,0)

                # HIỂN THỊ ảnh reference
                cv2.imshow("Matched person", reference_display)

            else:

                label = "Unknown"
                color = (0,0,255)

                # đóng cửa sổ nếu không match
                cv2.destroyWindow("Matched person")

            # vẽ bounding box
            cv2.rectangle(
                frame,
                (x,y),
                (x+w,y+h),
                color,
                2
            )

            # hiển thị similarity
            cv2.putText(
                frame,
                f"{label} {similarity:.2f}",
                (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                color,
                2
            )

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()

Press Q to quit
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/s